### 1.데이터 로딩 및 확인

In [2]:
import pandas as pd

#csv 파일불러오기
df = pd.read_csv('bccard_201906.csv')

#앞부분 5개
display(df.head(5))




,Unnamed: 0,REG_YYMM,MEGA_CTY_NO,MEGA_CTY_NM,CTY_RGN_NO,CTY_RGN_NM,ADMI_CTY_NO,ADMI_CTY_NM,MAIN_BUZ_CODE,MAIN_BUZ_DESC,...,CSTMR_GUBUN,CSTMR_MEGA_CTY_NO,CSTMR_MEGA_CTY_NM,CSTMR_CTY_RGN_NO,CSTMR_CTY_RGN_NM,SEX_CTGO_CD,AGE_VAL,FLC,AMT,CNT
0,0,201906,11,서울특별시,1162,관악구,11620585,낙성대동,80,음식,...,내국인,11,서울특별시,1162,관악구,2,30대,2,26284804,1892
1,1,201906,11,서울특별시,1159,동작구,11590560,상도4동,30,생활,...,내국인,11,서울특별시,1165,서초구,2,20대,1,109290,18
2,2,201906,11,서울특별시,1162,관악구,11620595,청룡동,30,생활,...,내국인,11,서울특별시,1162,관악구,1,20대,1,268850,52
3,3,201906,11,서울특별시,1144,마포구,11440660,서교동,80,음식,...,내국인,11,서울특별시,1138,은평구,1,20대,1,44174450,1790
4,4,201906,11,서울특별시,1120,성동구,11200550,사근동,80,음식,...,내국인,11,서울특별시,1120,성동구,1,20대,1,60338146,3536


In [5]:
#뒷부분
display(df.tail(5))


,Unnamed: 0,REG_YYMM,MEGA_CTY_NO,MEGA_CTY_NM,CTY_RGN_NO,CTY_RGN_NM,ADMI_CTY_NO,ADMI_CTY_NM,MAIN_BUZ_CODE,MAIN_BUZ_DESC,...,CSTMR_GUBUN,CSTMR_MEGA_CTY_NO,CSTMR_MEGA_CTY_NM,CSTMR_CTY_RGN_NO,CSTMR_CTY_RGN_NM,SEX_CTGO_CD,AGE_VAL,FLC,AMT,CNT
99996,99996,201906,11,서울특별시,1165,서초구,11650520,서초2동,30,생활,...,내국인,43,충청북도,4311,청주시,1,50대,4,50600,10
99997,99997,201906,11,서울특별시,1117,용산구,11170520,용산2가동,30,생활,...,내국인,11,서울특별시,1162,관악구,1,40대,2,38640,7
99998,99998,201906,11,서울특별시,1156,영등포구,11560535,영등포동,30,생활,...,내국인,28,인천광역시,2817,미추홀구,2,30대,2,340590,15
99999,99999,201906,11,서울특별시,1141,서대문구,11410585,신촌동,40,쇼핑,...,내국인,44,충청남도,4413,천안시,1,20대,2,117100,3
100000,100000,201906,11,서울특별시,1135,노원구,11350710,상계9동,20,문화,...,내국인,41,경기도,4115,의정부시,1,40대,3,5227500,7


In [6]:
#데이터수
print("데이터 형태:", df.shape)

데이터 형태: (100001, 24)


### 2.서울시 거주/비거주 고객 비교 분석

In [7]:
# 서울 거주 여부를 나타내는 파생 변수 생성 (True: 서울 거주, False: 비거주)
df['is_seoul'] = df['CSTMR_MEGA_CTY_NM'] == '서울특별시'

# 그룹별 분석 (고객 수, 총 소비액, 이용 건수 합계)
seoul_comparison = df.groupby('is_seoul').agg(
    고객수=('CSTMR_MEGA_CTY_NM', 'count'), 
    총소비액=('AMT', 'sum'),
    이용건수=('CNT', 'sum')
)

display(seoul_comparison)

,고객수,총소비액,이용건수
is_seoul,,,
False,45851,146587135822,4950200
True,54150,119663142676,5542462


### 3.편의점 소비 정보 분석

In [15]:
# 1) 전체 편의점 소비액
df_cvs = df[df['TP_BUZ_NO'] == 4010] # 데이터 타입이 문자형이라면 '4010'으로 변경하세요.
print("전체 편의점 소비액:", df_cvs['AMT'].sum())

# 2) 강남구 소비액 (업종 무관)
df_gangnam = df[df['MEGA_CTY_NM'] == '서울특별시']
print("서울특별시 전체 소비액:", df_gangnam['AMT'].sum())


전체 편의점 소비액: 7299184098
서울특별시 전체 소비액: 266250278498


In [13]:
# 3) 강남구 편의점 소비액
df_gangnam_cvs = df[(df['TP_BUZ_NO'] == 4010) & (df['CTY_RGN_NM'] == '강남구')]
print("강남구 편의점 소비액:", df_gangnam_cvs['AMT'].sum())

강남구 편의점 소비액: 707275140


In [18]:
# 4) 강남구 편의점 소비액 중 강남구 거주자와 비거주자의 소비액 비교
# 고객 시군구 거주지(CSTMR_CTY_RGN_NM)가 '강남구'인지 확인
df_gangnam_cvs = df_gangnam_cvs.copy()
df_gangnam_cvs['is_gangnam_resident'] = df_gangnam_cvs['CSTMR_CTY_RGN_NM'] == '강남구'

gangnam_cvs_comparison = df_gangnam_cvs.groupby('is_gangnam_resident').agg(
    소비액합계=('AMT', 'sum')
)
print("\n[강남구 편의점 소비 - 강남구 거주자(True) vs 비거주자(False) 비교]")
display(gangnam_cvs_comparison)


[강남구 편의점 소비 - 강남구 거주자(True) vs 비거주자(False) 비교]


,소비액합계
is_gangnam_resident,
False,418895250
True,288379890
